# Earnings Call Prediction — Data Demo

**Author:** Shannon Maccallum  
**Project:** Predicting Stock Returns from Earnings Call Transcripts

This notebook demonstrates the full data loading pipeline:
1. Load earnings call transcripts (Alpha Vantage API, with local caching)
2. Fetch post-earnings stock returns (Yahoo Finance)
3. Combine into a paired dataset
4. Show what the model **inputs** and **targets** look like

---

### Model Inputs and Targets
| Component | Description |
|-----------|-------------|
| **Input** | Earnings call transcript text (tokenized via FinBERT tokenizer) |
| **Target** | Post-earnings return (%) over a configurable window (1, 3, or 5 trading days) |

## 0. Setup

Install the package if you haven't already:
```bash
pip install -e .
```

Add your Alpha Vantage key to a `.env` file in the repo root:
```
ALPHA_VANTAGE_API_KEY=your_key_here
```

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from earnings_transcript_predictor import TranscriptLoader, PriceLoader, EarningsDataset
from earnings_transcript_predictor.utils import clean_transcript

print('Package imported successfully!')

## 1. Load Example Data (no API key needed)

The repo includes 3 pre-fetched example transcripts with their actual post-earnings returns.
This lets the instructor run the notebook without needing an API key.

In [ ]:
# Load the bundled example data points
examples_path = Path('../data/examples/example_transcripts.json')
with open(examples_path) as f:
    examples = json.load(f)

print(f'Loaded {len(examples)} example transcripts')
print('\nFields in each record:')
for key in examples[0].keys():
    print(f'  {key}: {type(examples[0][key]).__name__}')

In [ ]:
# Quick look at the examples as a DataFrame
df = pd.DataFrame([{
    'symbol': e['symbol'],
    'quarter': e['quarter'],
    'earnings_date': e['earnings_date'],
    'price_day0': e['price_day0'],
    'price_dayN': e['price_dayN'],
    'return_pct': e['return_pct'],
    'direction': e['direction'],
    'return_window': e['return_window'],
    'transcript_words': len(e['transcript'].split()),
} for e in examples])

df

**Observations:**
- `return_pct` is the **target variable** — what the model will predict
- `direction` is a derived binary label (1 = stock went up, 0 = down)
- Transcripts are ~150–200 words in the examples; real full transcripts are 5,000–15,000 words
- Returns vary a lot: META dropped 10.5% despite strong revenue — driven by capex guidance language

## 2. Fetch a Live Transcript (requires API key)

In [ ]:
# Uncomment and run if you have an API key in your .env file

# loader = TranscriptLoader()  # reads ALPHA_VANTAGE_API_KEY from .env
# transcript_data = loader.load('AAPL', '2024Q1')
# print(f"Symbol: {transcript_data['symbol']}")
# print(f"Quarter: {transcript_data['quarter']}")
# print(f"Transcript preview (first 500 chars):")
# print(transcript_data['transcript'][:500])

**Caching note:** After the first API call, the transcript is saved to `data/transcripts/AAPL_2024Q1.json`. Every subsequent call reads from disk — no API hit. This addresses the instructor's feedback about not re-hitting the API on every batch run.

## 3. Compute Post-Earnings Returns

In [ ]:
# Demonstrate the PriceLoader with a real ticker (uses Yahoo Finance, no API key needed)
price_loader = PriceLoader(return_window=3)

result = price_loader.get_return('AAPL', '2024-02-01')
print('Return data for AAPL Q1 2024 earnings:')
for k, v in result.items():
    print(f'  {k}: {v}')

In [ ]:
# Batch returns for all example tickers
pairs = [(e['symbol'], e['earnings_date']) for e in examples]
returns_df = price_loader.get_returns_batch(pairs)
print('Batch returns:')
returns_df

### Return Window Discussion

Per the instructor's feedback, the return window matters a lot:

| Window | What it captures | Consideration |
|--------|-----------------|---------------|
| **1 day** | Immediate market reaction | If call is after-hours, day 0 close already moves; this captures *next-day* drift |
| **3 days** | Short-term drift post-announcement | Most common in event studies; balances signal vs. noise |
| **5 days** | One full trading week | More noise from unrelated market events |

We default to **3 days** for now. The `PriceLoader(return_window=N)` makes it easy to experiment.

## 4. Text Preprocessing

In [ ]:
from earnings_transcript_predictor.utils import clean_transcript

raw = examples[0]['transcript']
cleaned = clean_transcript(raw)

print(f'Raw length:    {len(raw)} chars')
print(f'Cleaned length: {len(cleaned)} chars')
print(f'\nCleaned preview:')
print(cleaned[:400])

## 5. PyTorch Dataset — Model Inputs & Targets

In [ ]:
# Build the dataset from our example records
# In real training, 'examples' would be thousands of transcript+return dicts

dataset = EarningsDataset(examples, max_length=128)  # 128 for demo speed; use 512 for training
print(f'Dataset size: {len(dataset)} samples')

In [ ]:
# Inspect a single sample — this is exactly what the model will receive
sample = dataset[0]

print('=== MODEL INPUTS ===')
print(f'input_ids shape:      {sample["input_ids"].shape}   (one token ID per position)')
print(f'attention_mask shape: {sample["attention_mask"].shape}   (1=real token, 0=padding)')
print(f'\ninput_ids (first 20 tokens): {sample["input_ids"][:20]}')

print('\n=== MODEL TARGET ===')
print(f'return_pct (label):   {sample["label"]:.4f}%')

print('\n=== METADATA ===')
print(f'symbol:               {sample["symbol"]}')
print(f'quarter:              {sample["quarter"]}')
print(f'earnings_date:        {sample["earnings_date"]}')

**Summary of model I/O:**
- **Input:** `input_ids` tensor of shape `(512,)` — the tokenized transcript
- **Input:** `attention_mask` tensor of shape `(512,)` — tells the model which tokens are real vs. padding
- **Target:** `label` scalar — the post-earnings return % (e.g. `2.74` or `-10.56`)

## 6. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Return distribution
returns = [e['return_pct'] for e in examples]
axes[0].bar([e['symbol'] for e in examples], returns, color=['red' if r < 0 else 'green' for r in returns])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Post-Earnings 3-Day Returns (Example Data)')
axes[0].set_ylabel('Return (%)')
axes[0].set_xlabel('Ticker')

# Direction breakdown
directions = [e['direction'] for e in examples]
axes[1].pie(
    [sum(directions), len(directions) - sum(directions)],
    labels=['Up (1)', 'Down (0)'],
    colors=['green', 'red'],
    autopct='%1.0f%%',
)
axes[1].set_title('Direction Label Distribution')

plt.tight_layout()
plt.show()

print(f'Mean return: {pd.Series(returns).mean():.2f}%')
print(f'Std return:  {pd.Series(returns).std():.2f}%')
print('\nNote: With a full dataset (100s of transcripts), we expect high variance')
print('and roughly equal up/down split, consistent with the efficient market hypothesis.')

## Summary

This notebook demonstrated:
1. ✅ **TranscriptLoader** — fetches and caches Alpha Vantage transcripts
2. ✅ **PriceLoader** — computes configurable-window post-earnings returns from Yahoo Finance
3. ✅ **EarningsDataset** — PyTorch Dataset pairing tokenized transcripts with return targets
4. ✅ **Model inputs:** `input_ids` and `attention_mask` tensors of shape `(512,)`
5. ✅ **Model target:** `return_pct` float (percentage return over N trading days)

Next milestones will cover: full data collection, FinBERT fine-tuning, and evaluation.